# SENTRA AI — Deepfake Detector Training (FaceForensics++ + DFDC)

This notebook trains a **real, trainable** deepfake image classifier on two combined
datasets:

- **FaceForensics++** (real + manipulated face-swap videos)
- **DFDC** (DeepFake Detection Challenge sample videos, with `metadata.json` labels)

Unlike inference-only baseline notebooks, this one runs an **actual training loop**
(forward pass → loss → `backward()` → `optimizer.step()`), saves the best checkpoint,
and exports a model file you can plug straight into the SENTRA AI Flask app's
`models/image_detector.py`.

## How to use this notebook
1. Open a **new Kaggle Notebook**.
2. Under **Add Data**, attach:
   - The official Kaggle competition dataset **`deepfake-detection-challenge`** (DFDC)
   - A **FaceForensics++** dataset (search Kaggle Datasets for a community upload, e.g.
     `xdxd003/ff-c23` or similar — folder layout just needs `original_sequences` /
     `manipulated_sequences` subfolders; adjust `CONFIG['FF_PATH']` below to match).
3. Turn on **GPU** (Settings → Accelerator → GPU T4 x2 or P100).
4. Edit the paths in the `CONFIG` cell below to match your attached datasets.
5. Run All. Training checkpoints save to `/kaggle/working/`.
6. Download `best_model.pth` from the notebook's Output tab when done.
7. Follow the **Integration Guide** at the bottom to plug it into the Flask app.

> ⚠️ Real training here takes real time — expect 1–4 hours on a Kaggle GPU depending on
> how many frames per video you extract and how many epochs you run. Start with the
> small defaults below to sanity-check everything works, then scale up.


## 1. Install & Import Dependencies

In [ ]:
# Kaggle notebooks ship with torch, torchvision, opencv, pandas, sklearn pre-installed.
# timm gives us easy access to Xception / EfficientNet backbones with pretrained ImageNet weights.
!pip install -q timm facenet-pytorch

import os
import json
import random
import glob
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from sklearn.metrics import log_loss, accuracy_score, roc_auc_score, confusion_matrix
from tqdm.auto import tqdm
import timm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


## 2. Configuration — EDIT THESE PATHS to match your attached Kaggle datasets

In [ ]:
CONFIG = {
    # --- DFDC (Kaggle competition dataset) ---
    "DFDC_PATH": "/kaggle/input/deepfake-detection-challenge/train_sample_videos",
    "DFDC_METADATA": "/kaggle/input/deepfake-detection-challenge/train_sample_videos/metadata.json",

    # --- FaceForensics++ (community Kaggle dataset — adjust to your attached dataset's root) ---
    # Expected subfolders somewhere under this root:
    #   original_sequences/**/*.mp4        -> label REAL
    #   manipulated_sequences/**/*.mp4     -> label FAKE
    "FF_PATH": "/kaggle/input/faceforensics/faceforensics",

    # --- Face extraction ---
    "FRAMES_PER_VIDEO": 10,       # how many sampled frames to pull from each video
    "FACE_SIZE": 299,             # Xception expects 299x299; use 224 if you switch to EfficientNet
    "MAX_VIDEOS_PER_CLASS": 800,  # cap for a reasonable first training run; raise once it all works

    # --- Output ---
    "FACES_OUT_DIR": "/kaggle/working/faces",   # cached extracted face crops
    "CHECKPOINT_DIR": "/kaggle/working/checkpoints",

    # --- Training ---
    "BACKBONE": "xception",      # or "tf_efficientnet_b0_ns"
    "BATCH_SIZE": 32,
    "EPOCHS": 8,
    "LR": 1e-4,
    "VAL_SPLIT": 0.15,
    "NUM_WORKERS": 2,
}

os.makedirs(CONFIG["FACES_OUT_DIR"], exist_ok=True)
os.makedirs(CONFIG["CHECKPOINT_DIR"], exist_ok=True)
print(json.dumps(CONFIG, indent=2))


## 3. Face Detector

Uses OpenCV's DNN face detector when available, falling back to `facenet-pytorch`'s
MTCNN for robust face detection without needing dlib's C++ build (which is often the
slowest, most fragile part of offline Kaggle kernels).

In [ ]:
from facenet_pytorch import MTCNN

mtcnn = MTCNN(image_size=CONFIG["FACE_SIZE"], margin=20, post_process=False, device=DEVICE, keep_all=False)

def extract_face(frame_bgr):
    """Takes a BGR OpenCV frame, returns a cropped+aligned RGB face image (numpy array) or None."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    try:
        face_tensor = mtcnn(rgb)
    except Exception:
        return None
    if face_tensor is None:
        return None
    face_np = face_tensor.permute(1, 2, 0).byte().numpy()  # HWC, uint8
    return face_np


## 4. Build Video Lists — FaceForensics++

In [ ]:
def ff_build_video_list(ff_root):
    """Walks a FaceForensics++-style directory tree and returns [(video_path, label)]."""
    videos = []
    root = Path(ff_root)
    if not root.exists():
        print(f"[WARN] FF_PATH not found: {ff_root} — skipping FaceForensics++.")
        return videos

    real_videos = list(root.rglob("original_sequences/**/*.mp4"))
    fake_videos = list(root.rglob("manipulated_sequences/**/*.mp4"))

    random.shuffle(real_videos)
    random.shuffle(fake_videos)
    real_videos = real_videos[:CONFIG["MAX_VIDEOS_PER_CLASS"]]
    fake_videos = fake_videos[:CONFIG["MAX_VIDEOS_PER_CLASS"]]

    videos += [(str(v), 0) for v in real_videos]   # 0 = REAL
    videos += [(str(v), 1) for v in fake_videos]   # 1 = FAKE
    print(f"FaceForensics++: {len(real_videos)} real, {len(fake_videos)} fake videos found.")
    return videos

ff_videos = ff_build_video_list(CONFIG["FF_PATH"])


## 5. Build Video Lists — DFDC (uses metadata.json for labels)

In [ ]:
def dfdc_build_video_list(dfdc_dir, metadata_path):
    videos = []
    if not os.path.exists(metadata_path):
        print(f"[WARN] DFDC metadata not found: {metadata_path} — skipping DFDC.")
        return videos

    with open(metadata_path) as f:
        metadata = json.load(f)

    real_list, fake_list = [], []
    for fname, info in metadata.items():
        video_path = os.path.join(dfdc_dir, fname)
        if not os.path.exists(video_path):
            continue
        label = 0 if info.get("label") == "REAL" else 1
        (real_list if label == 0 else fake_list).append((video_path, label))

    random.shuffle(real_list)
    random.shuffle(fake_list)
    real_list = real_list[:CONFIG["MAX_VIDEOS_PER_CLASS"]]
    fake_list = fake_list[:CONFIG["MAX_VIDEOS_PER_CLASS"]]

    videos = real_list + fake_list
    print(f"DFDC: {len(real_list)} real, {len(fake_list)} fake videos found.")
    return videos

dfdc_videos = dfdc_build_video_list(CONFIG["DFDC_PATH"], CONFIG["DFDC_METADATA"])

all_videos = ff_videos + dfdc_videos
random.shuffle(all_videos)
print(f"\nTotal combined videos: {len(all_videos)}  "
      f"(real={sum(1 for _,l in all_videos if l==0)}, fake={sum(1 for _,l in all_videos if l==1)})")

assert len(all_videos) > 0, (
    "No videos found from either dataset. Double-check CONFIG['FF_PATH'] / CONFIG['DFDC_PATH'] "
    "match the datasets you attached under 'Add Data'."
)


## 6. Extract Faces From Videos (cached to disk)

This is the slowest one-time step: it samples frames from every video, runs face
detection, and writes cropped face JPEGs into `real/` and `fake/` folders so the
training loop below reads fast image files instead of re-decoding video every epoch.

In [ ]:
def extract_faces_from_videos(video_list, out_dir, frames_per_video):
    real_dir = os.path.join(out_dir, "real")
    fake_dir = os.path.join(out_dir, "fake")
    os.makedirs(real_dir, exist_ok=True)
    os.makedirs(fake_dir, exist_ok=True)

    saved = 0
    for video_path, label in tqdm(video_list, desc="Extracting faces"):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            cap.release()
            continue

        frame_indices = np.linspace(0, total_frames - 1, frames_per_video, dtype=int)
        video_id = Path(video_path).stem

        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, frame = cap.read()
            if not ok:
                continue
            face = extract_face(frame)
            if face is None:
                continue
            out_path = os.path.join(real_dir if label == 0 else fake_dir, f"{video_id}_{idx}.jpg")
            cv2.imwrite(out_path, cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
            saved += 1

        cap.release()

    print(f"Saved {saved} face crops to {out_dir}")

extract_faces_from_videos(all_videos, CONFIG["FACES_OUT_DIR"], CONFIG["FRAMES_PER_VIDEO"])


## 7. PyTorch Dataset + Train/Val Split

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        for label, sub in [(0, "real"), (1, "fake")]:
            for path in glob.glob(os.path.join(root_dir, sub, "*.jpg")):
                self.samples.append((path, label))
        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(img)
        return img, label


train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((CONFIG["FACE_SIZE"], CONFIG["FACE_SIZE"])),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((CONFIG["FACE_SIZE"], CONFIG["FACE_SIZE"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

full_dataset = FaceDataset(CONFIG["FACES_OUT_DIR"], transform=train_transform)
print(f"Total face crops available for training: {len(full_dataset)}")

val_size = int(len(full_dataset) * CONFIG["VAL_SPLIT"])
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                 generator=torch.Generator().manual_seed(SEED))
val_ds.dataset.transform = val_transform  # use non-augmented transform for validation

train_loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True,
                           num_workers=CONFIG["NUM_WORKERS"], pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False,
                         num_workers=CONFIG["NUM_WORKERS"], pin_memory=True)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")


## 8. Model — Xception / EfficientNet via `timm` (ImageNet-pretrained backbone, binary head)

In [ ]:
def build_model(backbone_name):
    model = timm.create_model(backbone_name, pretrained=True, num_classes=2)
    return model.to(DEVICE)

model = build_model(CONFIG["BACKBONE"])
print(f"Model: {CONFIG['BACKBONE']} — {sum(p.numel() for p in model.parameters()):,} parameters")


## 9. Training Loop — real forward/backward pass, checkpointing best model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=CONFIG["LR"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["EPOCHS"])

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for imgs, labels in tqdm(loader, desc="Train" if train else "Val", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            if train:
                optimizer.zero_grad()

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            probs = torch.softmax(outputs, dim=1)[:, 1]  # P(fake)
            preds = torch.argmax(outputs, dim=1)

            all_probs.extend(probs.detach().cpu().numpy())
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    try:
        ll = log_loss(all_labels, all_probs, labels=[0, 1])
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        ll, auc = float("nan"), float("nan")

    return avg_loss, acc, ll, auc


best_val_loss = float("inf")
history = []

for epoch in range(1, CONFIG["EPOCHS"] + 1):
    train_loss, train_acc, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_logloss, val_auc = run_epoch(val_loader, train=False)
    scheduler.step()

    print(f"Epoch {epoch}/{CONFIG['EPOCHS']} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_logloss={val_logloss:.4f} val_auc={val_auc:.4f}")

    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                     "val_loss": val_loss, "val_acc": val_acc,
                     "val_logloss": val_logloss, "val_auc": val_auc})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        ckpt_path = os.path.join(CONFIG["CHECKPOINT_DIR"], "best_model.pth")
        torch.save({
            "model_state_dict": model.state_dict(),
            "backbone": CONFIG["BACKBONE"],
            "face_size": CONFIG["FACE_SIZE"],
            "epoch": epoch,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }, ckpt_path)
        print(f"  ↳ New best model saved to {ckpt_path}")

history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(CONFIG["CHECKPOINT_DIR"], "training_history.csv"), index=False)
history_df


## 10. Final Evaluation on the Validation Set

In [ ]:
checkpoint = torch.load(os.path.join(CONFIG["CHECKPOINT_DIR"], "best_model.pth"), map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

val_loss, val_acc, val_logloss, val_auc = run_epoch(val_loader, train=False)
print(f"Best checkpoint (epoch {checkpoint['epoch']}):")
print(f"  Val Accuracy : {val_acc:.4f}")
print(f"  Val Log Loss : {val_logloss:.4f}")
print(f"  Val ROC AUC  : {val_auc:.4f}")

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        outputs = model(imgs.to(DEVICE))
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print("Confusion matrix [[TN, FP], [FN, TP]] (0=Real, 1=Fake):")
print(cm)


## 11. Export for Deployment

`best_model.pth` (already saved above) is everything the Flask app needs. Download it
from the notebook's **Output** panel after the run finishes, then follow the
integration guide below.

In [ ]:
print("Checkpoint ready at:", os.path.join(CONFIG["CHECKPOINT_DIR"], "best_model.pth"))
print("File size (MB):", os.path.getsize(os.path.join(CONFIG["CHECKPOINT_DIR"], "best_model.pth")) / 1e6)


## 12. Integration Guide — plugging `best_model.pth` into the SENTRA AI Flask app

1. Download `best_model.pth` from this notebook's Output tab.
2. In your SENTRA AI project, create a `models/weights/` folder and place the file there:
   `app/models/weights/best_model.pth`
3. Add `torch`, `torchvision`, and `timm` to `requirements.txt`.
4. Replace `app/models/image_detector.py` with the **real-model version** provided
   alongside this notebook (`image_detector_real.py`) — it automatically loads
   `best_model.pth` if present, and transparently falls back to the original dummy
   predictions if the weights file is missing, so the app never breaks.
5. Restart the Flask app (`python app.py`). Uploaded images will now be scored by your
   trained model instead of the placeholder logic.

For **video** and **NCII** detection, the same trained face classifier can be reused:
run it over multiple sampled frames (as the training pipeline already does) and
aggregate the per-frame fake-probability (mean or max) into a single video-level score
— this is exactly the pattern already stubbed out with `TODO`s in
`video_detector.py`.
